In [ ]:
# ======================================================
# Notebook: 8D Hyperparameter Optimisation (Hyperparameter tuning surrogate)
# Inputs: (40,8) | Output: (40,)
# Goal: maximise validation accuracy
# ======================================================

import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (40,8)
y = np.load("/mnt/data/initial_outputs.npy")     # (40,)

# Hyperparameter tuning
param_dist = {
    "n_estimators": [50,100,200,300,500],
    "max_depth": [None,3,5,8,12],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4],
    "max_features": ["sqrt","log2",None]
}

search = RandomizedSearchCV(
    RandomForestRegressor(),
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    random_state=42
)

search.fit(X, y)
model = search.best_estimator_

print("Best params:", search.best_params_)

# Candidate sampling
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(8)]
n_candidates = 10000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], n_candidates) for b in bounds
])

# Ensemble predictions
preds = np.array([tree.predict(X_grid) for tree in model.estimators_])
mean_pred = preds.mean(axis=0)
uncertainty = preds.std(axis=0)

# Acquisition
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,8)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,8) hyperparameter candidates:")
print(next_points)